Question 3 - Overfitting

In [ ]:

# higher epoches
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F



from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

# 2. Device Configuration and Seed Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import random

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

# 3. Load CIFAR-10 Dataset

#Basic Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

#Download Dataset
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

#Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# 4. CNN Starter Code
import torch
import torch.nn as nn
import torch.nn.functional as F

class InitialCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.fc1 = nn.Linear(256 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
# Changing the Activaition function from sigmoid to relu
        #x = torch.sigmoid(self.conv1(x))
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

    #    x = torch.sigmoid(self.conv2(x))
        x = torch.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

     #   x = torch.sigmoid(self.conv3(x))
        x = torch.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)

     #   x = torch.sigmoid(self.conv4(x))
        x = torch.relu(self.conv4(x))
        x = F.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)

      #  x = torch.sigmoid(self.fc1(x))
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# 5. MLP Starter Code
class InitialMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten

        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

# 6. Initialize Network
model = InitialCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# 7. Training Function
def train(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    gradient_norms = {}

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        # Save gradient norms
        for name, param in model.named_parameters():

            if param.grad is not None:

                grad_norm = param.grad.norm().item()

                if name not in gradient_norms:
                    gradient_norms[name] = []

                gradient_norms[name].append(grad_norm)

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total

    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy, gradient_norms

# 8. Testing Function
def test(model, loader):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total

    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy

# 9. Main Training Loop
num_epochs = 60

train_losses = []
test_losses = []

train_accs = []
test_accs = []

all_epochs_grad_norms = [] # To store gradient norms for all epochs

for epoch in range(num_epochs):

    train_loss, train_acc, grad_norms = train(
        model,
        train_loader
    )

    all_epochs_grad_norms.append(grad_norms) # Store grad_norms for the current epoch

    test_loss, test_acc = test(
        model,
        test_loader
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{num_epochs}")

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")

    print("-" * 40)

# 10. Train the MLP
mlp_model = InitialMLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=0.001
)

# 11. Example image visualization code
classes = train_dataset.classes

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2,5, figsize=(10,5))

for i, ax in enumerate(axes.flat):

    img = images[i] / 2 + 0.5
    img = img.permute(1,2,0)

    ax.imshow(img)
    ax.set_title(classes[labels[i]])
    ax.axis('off')

plt.show()

In [ ]:

# OVERFITTING DIAGNOSIS AND MITIGATION ARCHITECTURES

# 1. ARCHITECTURE DEFINITIONS

class SmallerCNN(nn.Module):
    """
    A low-capacity network with fewer convolutional channels and layers
    to inherently limit parameter memorization capability.
    """
    def __init__(self):
        super().__init__()
        # Reduced channels from (32, 64, 128, 256) down to a compact 3-layer configuration
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        # Spatial dimensions mapping: 32x32 -> MaxPool -> 16x16 -> MaxPool -> 8x8 -> MaxPool -> 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.max_pool2d(torch.relu(self.conv1(x)), 2)
        x = F.max_pool2d(torch.relu(self.conv2(x)), 2)
        x = F.max_pool2d(torch.relu(self.conv3(x)), 2)
        
        x = x.view(x.size(0), -1) # Flatten
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


class DropoutCNN(nn.Module):
    """
    The baseline InitialCNN architecture reinforced with Dropout layers 
    to break pixel co-adaptations and enforce robust generalization.
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.fc1 = nn.Linear(256 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, 10)
        
        # Regularization Dropout rates
        self.spatial_dropout = nn.Dropout2d(0.2) # Regularizes convolutional feature maps
        self.dense_dropout = nn.Dropout(0.4)     # Regularizes dense hidden units

    def forward(self, x):
        x = F.max_pool2d(torch.relu(self.conv1(x)), 2)
        x = self.spatial_dropout(x)
        
        x = F.max_pool2d(torch.relu(self.conv2(x)), 2)
        x = self.spatial_dropout(x)
        
        x = F.max_pool2d(torch.relu(self.conv3(x)), 2)
        
        x = F.max_pool2d(torch.relu(self.conv4(x)), 2)
        
        x = x.view(x.size(0), -1) # Flatten
        x = torch.relu(self.fc1(x))
        x = self.dense_dropout(x)
        
        return self.fc2(x)

# 2. MODEL TRAINING LOOPS (COMPATIBLE WITH ORIGINAL SCRIPT STRUCTURE)

num_epochs = 30 # 30 epochs are ideal to capture clear convergence trends
criterion = nn.CrossEntropyLoss()

# --- Initialize and Train Smaller CNN ---
smaller_model = SmallerCNN().to(device)
smaller_optimizer = optim.Adam(smaller_model.parameters(), lr=0.001)

smaller_train_losses, smaller_test_losses = [], []
smaller_train_accs, smaller_test_accs = [], []

print("Training Smaller Model Architecture...")
for epoch in range(num_epochs):
    # CRITICAL: We safely point the global variable 'optimizer' to our smaller optimizer 
    # right before running the original train() function
    optimizer = smaller_optimizer
    
    t_loss, t_acc, _ = train(smaller_model, train_loader)
    v_loss, v_acc = test(smaller_model, test_loader)
    
    smaller_train_losses.append(t_loss)
    smaller_test_losses.append(v_loss)
    smaller_train_accs.append(t_acc)
    smaller_test_accs.append(v_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs} -> Train Acc: {t_acc:.2f}% | Test Acc: {v_acc:.2f}%")

# --- Initialize and Train Dropout CNN ---
dropout_model = DropoutCNN().to(device)
dropout_optimizer = optim.Adam(dropout_model.parameters(), lr=0.001)

dropout_train_losses, dropout_test_losses = [], []
dropout_train_accs, dropout_test_accs = [], []

print("\nTraining Dropout (Regularized) Model Architecture...")
for epoch in range(num_epochs):
    # CRITICAL: Dynamically update the global 'optimizer' variable to target our dropout optimizer
    optimizer = dropout_optimizer
    
    t_loss, t_acc, _ = train(dropout_model, train_loader)
    v_loss, v_acc = test(dropout_model, test_loader)
    
    dropout_train_losses.append(t_loss)
    dropout_test_losses.append(v_loss)
    dropout_train_accs.append(t_acc)
    dropout_test_accs.append(v_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{num_epochs} -> "
              f"Train Loss: {t_loss:.4f} | Test Loss: {v_loss:.4f} | "
              f"Train Acc: {t_acc:.2f}% | Test Acc: {v_acc:.2f}%")
        

In [ ]:
# Vis.
# 3. COMPREHENSIVE LEARNING CURVES PLOTTING BLOCK

epochs_range = range(1, num_epochs + 1)
baseline_epochs = range(1, len(train_losses) + 1) # Captures your 60-epoch baseline arrays

# Setup a clean, 3x2 multi-panel figure plot layout
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# --- ROW 1: Baseline Overfitted CNN (Uses your existing local train_losses/test_losses) ---
axes[0, 0].plot(baseline_epochs, train_losses, label='Train Loss', color='#1f77b4', linewidth=2)
axes[0, 0].plot(baseline_epochs, test_losses, label='Test Loss', color='#ff7f0e', linewidth=2)
axes[0, 0].set_title('Graph 9: Baseline Overfitted CNN: Loss Curves', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, linestyle=':', alpha=0.6)
axes[0, 0].legend()

axes[0, 1].plot(baseline_epochs, train_accs, label='Train Acc', color='#2ca02c', linewidth=2)
axes[0, 1].plot(baseline_epochs, test_accs, label='Test Acc', color='#d62728', linewidth=2)
axes[0, 1].set_title('Graph 10: Baseline Overfitted CNN: Accuracy Curves', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].grid(True, linestyle=':', alpha=0.6)
axes[0, 1].legend()

#  ROW 2: Smaller Capacity CNN 
axes[1, 0].plot(epochs_range, smaller_train_losses, label='Train Loss', color='#1f77b4', linewidth=2)
axes[1, 0].plot(epochs_range, smaller_test_losses, label='Test Loss', color='#ff7f0e', linewidth=2)
axes[1, 0].set_title('Graph 11: Smaller Model Architecture: Loss Curves', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True, linestyle=':', alpha=0.6)
axes[1, 0].legend()

axes[1, 1].plot(epochs_range, smaller_train_accs, label='Train Acc', color='#2ca02c', linewidth=2)
axes[1, 1].plot(epochs_range, smaller_test_accs, label='Test Acc', color='#d62728', linewidth=2)
axes[1, 1].set_title('Graph 12: Smaller Model Architecture: Accuracy Curves', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epochs')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].grid(True, linestyle=':', alpha=0.6)
axes[1, 1].legend()

# --- ROW 3: Regularized Model with Dropout ---
axes[2, 0].plot(epochs_range, dropout_train_losses, label='Train Loss', color='#1f77b4', linewidth=2)
axes[2, 0].plot(epochs_range, dropout_test_losses, label='Test Loss', color='#ff7f0e', linewidth=2)
axes[2, 0].set_title('Graph 13: Dropout Regularization Model: Loss Curves', fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel('Epochs')
axes[2, 0].set_ylabel('Loss')
axes[2, 0].grid(True, linestyle=':', alpha=0.6)
axes[2, 0].legend()

axes[2, 1].plot(epochs_range, dropout_train_accs, label='Train Acc', color='#2ca02c', linewidth=2)
axes[2, 1].plot(epochs_range, dropout_test_accs, label='Test Acc', color='#d62728', linewidth=2)
axes[2, 1].set_title('Graph 14: Dropout Regularization Model: Accuracy Curves', fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel('Epochs')
axes[2, 1].set_ylabel('Accuracy (%)')
axes[2, 1].grid(True, linestyle=':', alpha=0.6)
axes[2, 1].legend()

plt.tight_layout()
plt.show()